# **Logistics Operations RAG Assistant**

## **Objective**

The main objective of this assignment is to process and analyse a collection of logistics and operational documents from AeroVelo Logistics Global to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Logistics Data: Gain an understanding of the structure, content, and context of the provided documents.
* Perform Exploratory Analysis: Analyse the documents to understand their length, distribution, and content.
* Prepare Documents for Retrieval: Clean and split the documents into smaller chunks suitable for semantic retrieval.
* Build a RAG Knowledge Base: Generate embeddings and store document chunks in a vector database.
* Retrieve Relevant Information: Retrieve the most relevant document chunks for a user query.
* Generate Grounded Responses: Use a Large Language Model (LLM) to answer questions based on the retrieved context.
* Evaluate the RAG System: Assess retrieval quality, answer correctness, grounding, and unsupported question handling.

### **Business Value**

The project aims to use RAG to improve access to logistics and operational information within AeroVelo Logistics Global. The key business objectives include:

* Faster Information Retrieval: <br>
Reduce the time required to search across logistics, supply chain, finance, governance, compliance, legal, and HR documents.

* Improved Operational Support: <br>
Provide relevant information for supplier, procurement, incident management, and operational queries.

* Consistent Responses: <br>
Generate answers grounded in approved organisational documents rather than relying only on the LLM's internal knowledge.

* Reduced Manual Search Effort: <br>
Allow users to query multiple documents through a single assistant instead of manually reviewing individual files.

* Better Handling of Unsupported Queries: <br>
Identify when the available document collection does not contain sufficient information to answer a question.

## **1. Data Loading, Preparation and Analysis** <font color=red>[40 marks]</font>

### **1.1 Data Understanding**

The dataset contains documents from AeroVelo Logistics Global covering different operational and business functions.

The documents are organised into the following folders:

- `compliance`: regulatory and compliance-related documents
- `finance`: financial authorisation and procurement-related documents
- `governance`: regional governance information
- `hr`: human resources policies
- `legal`: legal and supplier data-processing documents
- `operations`: logistics incident management procedures
- `pdfs`: documents available in PDF format
- `supply_chain`: supplier qualification and Bill of Materials information

The dataset contains both Markdown (`.md`) and PDF (`.pdf`) files. These documents will be loaded and converted into LangChain document objects while preserving their source information as metadata.

### **1.2 Load and Preprocess the Data** <font color=red>[12 marks]</font>

#### Loading libraries

In [3]:
## The following libraries might be useful
%pip install -q pandas matplotlib tqdm pypdf nltk scikit-learn
%pip install -U -q langchain langchain-core langchain-community
%pip install -U -q langchain-text-splitters langchain-huggingface langchain-chroma
%pip install -U -q sentence-transformers
%pip install -U -q langchain-ollama


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import essential libraries

import os
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from collections import Counter
from nltk.corpus import stopwords


#### **1.2.1 Load Logistics Documents** <font color=red>[7 marks]</font>

Load all Markdown (`.md`) and PDF (`.pdf`) documents from the dataset folder. Store useful metadata such as the file name, business category, file type, and source path for later retrieval and evaluation.

In [2]:
# Load all Markdown and PDF files from the dataset folder
root = Path("AeroVelo Logistics Global")
files = sorted(
    p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {".md", ".pdf"}
)
len(files), [p.name for p in files]


(10,
 ['SOP-COM-015_Regulatory_Breach_Report.md',
  'FIN-AUTH-101_Authorization_Framework.md',
  'RGM-2026_Regional_Governance_Matrix.md',
  'HR-POL-030-EMEA_Remote_Work.md',
  'HR-POL-030_Remote_Work_US.md',
  'LEG-GDPR-007_Supplier_Data_Processing.md',
  'OP-204_Incident_Management_v4.md',
  'RGM-2026_Regional_Governance_Matrix.pdf',
  'AVL-PUR-012_Supplier_Qualification.md',
  'BOM-FLT-H2NEX.md'])

In [3]:
# Load the logistics documents and store useful metadata
def read_text(path):
    if path.suffix.lower() == ".md":
        return path.read_text(encoding="utf-8")
    reader = PdfReader(str(path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

docs = []
for path in files:
    docs.append(Document(
        page_content=read_text(path),
        metadata={
            "file_name": path.name,
            "category": path.parent.name,
            "file_type": path.suffix.lower().lstrip("."),
            "source": str(path),
        },
    ))
len(docs), docs[0].metadata


(10,
 {'file_name': 'SOP-COM-015_Regulatory_Breach_Report.md',
  'category': 'compliance',
  'file_type': 'md',
  'source': 'AeroVelo Logistics Global/compliance/SOP-COM-015_Regulatory_Breach_Report.md'})

In [4]:
# Inspect the first loaded document
doc = docs[0]
doc.metadata, len(doc.page_content), doc.page_content[:400]


({'file_name': 'SOP-COM-015_Regulatory_Breach_Report.md',
  'category': 'compliance',
  'file_type': 'md',
  'source': 'AeroVelo Logistics Global/compliance/SOP-COM-015_Regulatory_Breach_Report.md'},
 2991,
 '# AeroVelo Logistics Global — Regulatory Breach Reporting\n\n**Document ID:** SOP-COM-015  \n**Version:** 2.0 (Current)  \n**Effective Date:** 2025-03-01  \n**Classification:** Internal — Compliance Officers and Above  \n**Owner:** Global Compliance Directorate  \n**Approved By:** General Counsel and Board Risk Committee\n\n---\n\n## 1. Purpose\n\nThis procedure defines the reporting obligations, escalation ti')

#### **1.2.2 Clean and Normalise Document Text** <font color=red>[5 marks]</font>

Clean the loaded document text by removing unnecessary whitespace and formatting inconsistencies while preserving the original document meaning and metadata.

In [5]:
# Clean and normalise the document text
def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for doc in docs:
    doc.page_content = clean_text(doc.page_content)

len(docs), len(docs[0].page_content)


(10, 2980)

### **1.3 Exploratory Data Analysis** <font color=red>[16 marks]</font>

#### **1.3.1 Analyse Document Lengths** <font color=red>[4 marks]</font>

Calculate the average, maximum and minimum document length.

In [6]:
# Calculate the average, maximum and minimum document length
lengths = [len(doc.page_content) for doc in docs]
length_stats = {
    "average": sum(lengths) / len(lengths),
    "maximum": max(lengths),
    "minimum": min(lengths),
}
length_stats


{'average': 4073.2, 'maximum': 5599, 'minimum': 1394}

#### **1.3.2 Analyse Word Frequencies** <font color=red>[6 marks]</font>

Analyse the frequency of words in the logistics documents and identify the most and least frequently occurring words.

In [7]:
# Analyse word frequencies in the logistics documents
stop = set(stopwords.words("english"))
tokens = []
for doc in docs:
    words = re.findall(r"[A-Za-z]+", doc.page_content.lower())
    tokens.extend(w for w in words if w not in stop and len(w) > 1)

counts = Counter(tokens)
most_common = counts.most_common(10)
least_n = min(counts.values())
least_common = sorted(w for w, n in counts.items() if n == least_n)
most_common, least_n, len(least_common), least_common[:15]


([('regional', 67),
  ('avl', 55),
  ('remote', 54),
  ('document', 46),
  ('operations', 40),
  ('global', 36),
  ('emea', 34),
  ('incident', 34),
  ('logistics', 29),
  ('work', 29)],
 1,
 372,
 ['acceptable',
  'acceptance',
  'accesses',
  'accessories',
  'adaptation',
  'additionally',
  'adherence',
  'administrative',
  'affected',
  'africa',
  'aggregate',
  'agvs',
  'alc',
  'allocated',
  'allowed'])

#### **1.3.3 Compare Document Similarity** <font color=red>[6 marks]</font>

Analyse the similarity between different logistics documents using TF-IDF vectors and cosine similarity.

In [8]:
# Transform documents using TF-IDF and compute document similarity
names = [d.metadata["file_name"] for d in docs]
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(d.page_content for d in docs)
sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim, index=names, columns=names)

pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append((sim[i, j], names[i], names[j]))
pairs.sort(reverse=True)
sim_df.round(3), pairs[0]


(                                          SOP-COM-015_Regulatory_Breach_Report.md  \
 SOP-COM-015_Regulatory_Breach_Report.md                                     1.000   
 FIN-AUTH-101_Authorization_Framework.md                                     0.197   
 RGM-2026_Regional_Governance_Matrix.md                                      0.222   
 HR-POL-030-EMEA_Remote_Work.md                                              0.108   
 HR-POL-030_Remote_Work_US.md                                                0.080   
 LEG-GDPR-007_Supplier_Data_Processing.md                                    0.264   
 OP-204_Incident_Management_v4.md                                            0.229   
 RGM-2026_Regional_Governance_Matrix.pdf                                     0.285   
 AVL-PUR-012_Supplier_Qualification.md                                       0.092   
 BOM-FLT-H2NEX.md                                                            0.064   
 
                                           FIN-AUTH-

### **1.4 Document Creation and Chunking** <font color=red>[12 marks]</font>

#### **1.4.1 Create Overlapping Document Chunks** <font color=red>[12 marks]</font>

Perform appropriate steps to split the logistics documents into smaller overlapping chunks while preserving the document metadata.

In [9]:
# Define a function to split documents into overlapping chunks
CHUNK_SIZE, CHUNK_OVERLAP = 800, 120

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(documents)

CHUNK_SIZE, CHUNK_OVERLAP


(800, 120)

In [10]:
# Process all documents and generate chunks
chunks = split_documents(docs)



In [11]:
# Check the total number of chunks
len(chunks)



68

In [ ]:
# Inspect a sample chunk


## **2. Vector Database and RAG Chain Creation** <font color=red>[35 marks]</font>

### **2.1 Vector Embeddings and Database Creation** <font color=red>[16 marks]</font>

#### **2.1.1 Initialise the Embedding Model** <font color=red>[5 marks]</font>

Initialise an embedding function to generate vector representations of the document chunks.

Initialise an embedding model to convert the document chunks into numerical vector representations for storage and similarity search in the vector database.

In [ ]:
# Initialise the embedding model


#### **2.1.2 Create and Populate the Vector Database** <font color=red>[11 marks]</font>

Load the document chunk embeddings into a Chroma vector database for semantic retrieval.

In [ ]:
# Create a function to add chunks to the vector database


In [ ]:
# Create and populate the Chroma vector database


In [ ]:
# Check the number of vectors stored


### **2.2 Create the RAG Chain** <font color=red>[19 marks]</font>

#### **2.2.1 Build the RAG Pipeline** <font color=red>[12 marks]</font>

Form the complete RAG pipeline by connecting the vector database retriever, prompt and language model.

In [ ]:
# Additional imports for the RAG chain are already included above


In [ ]:
# Format retrieved documents as context


In [ ]:
# Create the RAG chain


In [ ]:
# Initialise the RAG chain


#### **2.2.2 Generate Answers and Display Sources** <font color=red>[7 marks]</font>

Create a function to generate answers for logistics-related questions.

Use the RAG pipeline to generate an answer and display the source documents used for retrieval.

In [ ]:
# Create a function for question answering


In [ ]:
# Test the RAG assistant with an example logistics question


## **3. RAG Evaluation** <font color=red>[20 marks]</font>

### **3.1 Evaluation and Inference** <font color=red>[20 marks]</font>

#### **3.1.1 Prepare Evaluation Questions and Reference Answers** <font color=red>[4 marks]</font>

Create a small evaluation set containing logistics questions and their expected ground-truth answers.

In [ ]:
# Create evaluation questions and ground-truth answers


#### **3.1.2 Evaluate Answer and Retrieval Similarity** <font color=red>[10 marks]</font>

Create a function to evaluate the generated answers.

Evaluate the RAG responses using semantic similarity between the generated answer and ground truth, along with the relevance of the retrieved context.

In [ ]:
# Create a function to evaluate the RAG pipeline


In [ ]:
# Evaluate the RAG pipeline


#### **3.1.3 Interpret Results and Test Unsupported Questions** <font color=red>[6 marks]</font>

Draw inferences from the evaluation results and test how the RAG system handles a question that is not supported by the provided documents.

Summarise the average evaluation scores and verify that the assistant does not generate unsupported information when the answer is unavailable in the logistics knowledge base.

In [ ]:
# Summarise the evaluation results


In [ ]:
# Test an unsupported question


## **4. Conclusion** <font color=red>[5 marks]</font>

### **4.1 Conclusions and Insights** <font color=red>[5 marks]</font>

#### **4.1.1 Summarise Findings and Limitations** <font color=red>[5 marks]</font>

Conclude with the results obtained from the data analysis, document chunking, vector retrieval, RAG pipeline and evaluation.